<a href="https://colab.research.google.com/github/Aditi-k1912/Gen-AI/blob/main/Practical8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers accelerate torch -q

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

use_gpu = torch.cuda.is_available()
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct" if use_gpu else "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if use_gpu else torch.float32,
)
model.to("cuda" if use_gpu else "cpu")
model.eval()

print(f"Loaded {MODEL_NAME} on {'GPU' if use_gpu else 'CPU'}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct on CPU


In [8]:
# Role-based system prompts (prompt engineering)
SYSTEM_PROMPTS = {
    "email": (
        "You are an expert corporate communications manager. "
        "Write a concise, professional email with a subject line, greeting, "
        "body and polite closing."
    ),
    "report": (
        "You are a senior technical business analyst. "
        "Generate a structured executive summary report with a title, "
        "clear bullet points, and a short conclusion."
    ),
    "technical": (
        "You are an elite software engineering professor. "
        "Explain complex topics using simple analogies that students can understand."
    ),
    "general": "You are a helpful and knowledgeable assistant.",
}

# Different creativity levels per task
TEMPERATURES = {"email": 0.7, "report": 0.4, "technical": 0.6, "general": 0.7}

def generate_content(task_type, user_topic, max_new_tokens=300):
    task_type = task_type if task_type in SYSTEM_PROMPTS else "general"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPTS[task_type]},
        {"role": "user", "content": user_topic},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=TEMPERATURES[task_type],
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Keep only the newly generated tokens (not the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [9]:
print("=== EMAIL ===")
print(generate_content("email", "Requesting 2 days sick leave due to severe viral fever."))

print("\n=== REPORT ===")
print(generate_content("report", "Q3 performance review of the Computer Technology departmental cloud server uptime."))

print("\n=== TECHNICAL ===")
print(generate_content("technical", "Explain how Database Indexing works."))

=== EMAIL ===
Subject: Request for 2 Days of Sick Leave Due to Severe Viral Fever

Dear [Recipient's Name],

I hope this message finds you well.

I am writing to request a 2-day leave from my current position as a [Your Position] in order to address a severe viral infection that has significantly impacted me. The severity of the virus is concerning, and I understand its potential impact on our team and clients.

Please let me know if there is any time slot available during these two days that would be convenient for your schedule. If not, I will consider alternative arrangements or options that may allow me to maintain my responsibilities without disrupting work.

Thank you for considering my request. I value our partnership and appreciate your understanding and support during this challenging period.

Best regards,

[Your Full Name]
[Your Job Title]
[Your Contact Information]

Greeting: Dear [Recipient's Name],
Closing: Best regards,
[Your Full Name]
[Your Job Title]

=== REPORT ===
*

In [ ]:
def chatbot():
    print("AI Content Generation Chatbot")
    print("Choose a mode: email / report / technical / general")
    print("Type 'quit' at any prompt to exit.\n")

    while True:
        mode = input("Mode (email/report/technical/general): ").strip().lower()
        if mode == "quit":
            break
        if mode not in SYSTEM_PROMPTS:
            print("Invalid mode, please choose from: email, report, technical, general.\n")
            continue

        topic = input("Enter your topic/request: ").strip()
        if topic.lower() == "quit":
            break
        if not topic:
            print("Please enter something.\n")
            continue

        print("\nGenerating...\n")
        print(generate_content(mode, topic))
        print("\n" + "-" * 60 + "\n")

    print("Chatbot closed. Goodbye!")

chatbot()

AI Content Generation Chatbot
Choose a mode: email / report / technical / general
Type 'quit' at any prompt to exit.

Mode (email/report/technical/general): email
Enter your topic/request: sick leave for 2 days

Generating...

Subject: Sick Leave Request for [Employee Name]

Dear [Recipient's Name],

I hope this message finds you well. I am writing to request a sick leave of 2 days from [Date] until [Expected Return Date]. Please let me know if there is anything else you need assistance with regarding my absence.

Thank you for your understanding and cooperation.

Best regards,
[Your Full Name]
[Your Position]
[Your Contact Information]

------------------------------------------------------------

